In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#xii time sampled

mass = 1.0
k_const = 1.0
x_0 = 0.7
v_0 = 1.2
xi = 0.3

In [3]:
import torch
def sample_interior_points(N, T, device):
    """
    Sample N interior collocation points in:
        z ∈ (0, T)
        xi ∈ [0.1, 0.4]
    Returns:
        z_i, xi_i  (both shape [N,1])
    """
    z_i = torch.rand(N, 1, device=device) * T
    #xi_i = 0.1 + 0.3 * torch.rand(N, 1, device=device)
    return z_i

def sample_boundary_points(Nb, device):
    """
    Boundary = initial condition at z = 0
    xi ∈ [0.1, 0.4]
    """
    z_b = torch.zeros(Nb, 1, device=device)
    #xi_b = 0.1 + 0.3 * torch.rand(Nb, 1, device=device)
    return z_b

In [4]:
import torch.nn as nn
import torch.optim as optim
import math

device = "cuda" if torch.cuda.is_available() else "cpu"


In [5]:
class Sine(nn.Module):
    def __init__(self, w0=1.0):
        super().__init__()
        self.w0 = w0

    def forward(self, x):
        return torch.sin(self.w0 * x)

In [21]:
class PINN(nn.Module):
    def __init__(self, in_dim=1, width=64, depth=4, out_dim=1, w0=30.0, x0 = 0.7, v0 = 1.2):
        super().__init__()
        self.x0 = x0
        self.v0 = v0
        sefl.w0 = w0
        layers = []

        # ----- first layer -----
        layers.append(nn.Linear(in_dim, width))
        layers.append(Sine(w0))

        # ----- hidden layers -----
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(Sine(w0))   # hidden layers use w0 = 1

        # ----- final linear layer -----
        layers.append(nn.Linear(width, out_dim))

        self.net = nn.Sequential(*layers)

        # ----- SIREN initialization -----
        self.init_weights(w0)

    # ----------------------------
    # SIREN weight initialization
    # ----------------------------
    def init_weights(self, w0):
        with torch.no_grad():
            for i, m in enumerate(self.net):
                if isinstance(m, nn.Linear):
                    in_dim = m.weight.size(1)

                    if i == 0:
                        # first layer initialization (special)
                        m.weight.uniform_(-1 / in_dim, 1 / in_dim)
                    else:
                        # hidden layers initialization
                        bound = math.sqrt(6 / in_dim) / w0
                        m.weight.uniform_(-bound, bound)

                    nn.init.zeros_(m.bias)

    # ----------------------------
    # forward pass
    # ----------------------------
    def forward(self, z):
        inp = torch.cat([z], dim=1)

        N = self.net(inp)          # unconstrained NN output

        # ---- hard initial-condition constraint ----
        x = self.x0 + self.v0 * z + (z ** 2) * N

        return x

In [16]:
def derivatives(model, z):
    z = z.clone().detach().requires_grad_(True)

    x = model(z)

    dx = torch.autograd.grad(
        x, z, torch.ones_like(x), create_graph=True
    )[0]

    ddx = torch.autograd.grad(
        dx, z, torch.ones_like(dx), create_graph=True
    )[0]

    return x, dx, ddx


In [17]:
def ode_loss(model, z):
    x, dx, ddx = derivatives(model, z)
    residual = ddx + 2 * xi * dx + x
    return torch.mean(residual**2)


In [ ]:
# def ic_loss(model, z_b, xi_b, x0, v0):
#     x, dx, _ = derivatives(model, z_b, xi_b)

#     loss_x = torch.mean((x - x0) ** 2)
#     loss_v = torch.mean((dx - v0) ** 2)

#     return loss_x + loss_v


In [18]:
def total_loss(model, z_i, z_b, x0, v0, lam_ic=10.0):
    return ode_loss(model, z_i)

In [19]:
model = PINN().to(device)

In [23]:
time_bounds = [1, 2, 4, 8, 16, 20]
march_losses = []

print("Initial weight norm:",
      sum(p.norm().item() for p in model.parameters()))

for T in time_bounds:

    print(f"\n=== Training on domain [0, {T}] ===")

    # ----- resample collocation for new domain -----
    z_i = sample_interior_points(N=5000, T=T, device=device)
    z_b = sample_boundary_points(Nb=200, device=device)

    # ----- Adam warm-start training -----
    adam = torch.optim.Adam(model.parameters(), lr=1e-4)

    for step in range(5000):   # small retrain each stage
        adam.zero_grad()
        loss = total_loss(model, z_i, z_b, x_0, v_0)
        loss.backward()
        adam.step()

    print(f"Loss after adam: {T}: {loss.item():.3e}")
    # ----- L-BFGS refinement -----
    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=500,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        history_size=100,
        line_search_fn="strong_wolfe",
    )

    def closure():
        lbfgs.zero_grad()
        loss = total_loss(model, z_i, z_b, x_0, v_0)
        loss.backward()
        return loss

    loss = lbfgs.step(closure)

    march_losses.append(loss.item())
    print(f"Final loss at T={T}: {loss.item():.3e}")

    print("Weight norm after T =", T,
      sum(p.norm().item() for p in model.parameters()))



Initial weight norm: 6.179246285930276

=== Training on domain [0, 1] ===


KeyboardInterrupt: 